In [ ]:
! pip install transformers accelerate trl datasets bitsandbytes torch peft evaluate

In [ ]:
! pip uninstall -y torchao

**LOADING DATASET AND MAPPING **

In [ ]:
SYSTEM_PROMPT = """You check employee expenses against company policy.
Read the expense text and reply with only a JSON object with these keys:
amount, currency, category, policy_limit_amount, reimbursement_allowed, receipt_provided, flag, reason.

Categories: Travel, Ground Transport, Lodging, Meals, Software, Office Supplies, Training, Other.
Currencies: USD, EUR, GBP, INR, CAD.

Limits (USD / EUR / GBP / INR / CAD):
- Meals, per person: 75 / 70 / 60 / 6000 / 100
- Lodging, per night: 250 / 230 / 200 / 20000 / 340
- Ground Transport, per expense: 100 / 90 / 80 / 8000 / 135
- Travel, per expense: 800 / 750 / 650 / 64000 / 1100
- Software, per expense: 200 / 180 / 160 / 16000 / 270
- Office Supplies, per expense: 150 / 140 / 120 / 12000 / 200
- Training, per expense: 1000 / 900 / 800 / 80000 / 1350
- Other, per expense: 100 / 90 / 80 / 8000 / 135
A receipt is required above 25 USD, 25 EUR, 20 GBP, 2000 INR or 35 CAD.

Rules:
- policy_limit_amount = per-person limit x attendees for Meals, per-night limit x nights for Lodging, otherwise the limit. Use null if it cannot be worked out, and 0 if the expense is not reimbursable.
- receipt_provided is "No" only if the text says the receipt is missing.
- Not reimbursable: alcohol-only purchases, fines, upgrades, travel or rooms for family or friends, personal trips, hotel extras, personal subscriptions, gift cards, personal or home items, hobby classes, personal services or purchases.
- flag, first match wins: not_reimbursable; missing_info (amount, attendees, nights or a required receipt missing); over_limit (amount above policy_limit_amount); none.
- reimbursement_allowed is "No" only when the flag is not_reimbursable.
- Use null for anything the text does not state. reason briefly explains the flag."""

USER_TEMPLATE = "Expense: {text}"

In [ ]:
from datasets import load_dataset
import json

dataset = load_dataset("csv", data_files = "/expense_dataset_5000.csv",split = "train")
dataset = dataset.select(range(5000))

In [ ]:
def expense_classification_maptemplate(example):
  answer = json.dumps(
      {"amount":example["Amount"],"Currency": example["Currency"],"category": example["Category"], "policy_limit_amount": example["Policy_limit_amount"], "Reimbursement_allowed": example["Reimbursement_allowed"],"Receipt_provided":example["Receipt_provided"],"flag": example["Flag"], "Reason": example["Reason"]},
      ensure_ascii=False,)
  return {
      "prompt": [
          {"role": "system", "content": SYSTEM_PROMPT},
          {"role": "user", "content": USER_TEMPLATE.format(text=example["Text"])},
        ],
        "completion": [
            {"role": "assistant", "content": answer},
        ],
    }

In [ ]:
# load dataset

dataset = dataset.map(expense_classification_maptemplate,remove_columns=dataset.features)
dataset = dataset.train_test_split(test_size=0.2, shuffle=True, seed=42)

MODEL CALLING AND QUANTIZATION

In [ ]:
# load the model

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
dtype = torch.float16

# quantization

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype
)

# model calling

model = AutoModelForCausalLM.from_pretrained(model_id, quantization_config=bnb_config, device_map="auto")



LORA CONFIGURATION

In [ ]:
# weights are stored in bfloat16 and we are converting it to float16 as the gpu in colab supports float16


# weights are stored in bfloat16 and we are converting it to float16 as the gpu in colab supports float16
for p in model.parameters():
  if p.dtype == torch.bfloat16:
    p.data = p.data.to(torch.float16)

print(p.dtype)

# biases
for p in model.buffers():
  if p.dtype == torch.bfloat16:
    p.data = p.data.to(torch.float16)
print(p.dtype)

model.gradient_checkpointing_enable()
model.enable_input_require_grads()


In [ ]:
# LoRA Configuration

LoRA_config = LoraConfig(
    r = 16,
    lora_alpha = 32,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout = 0.05,
    bias = "none",
    task_type = "CAUSAL_LM"
)

In [ ]:
from trl import SFTTrainer, SFTConfig

args = SFTConfig(
    output_dir="FineTuning_ExpenseClassification_5k",
    num_train_epochs = 3,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    fp16=True,
    bf16=False,
    push_to_hub=False,
    report_to="tensorboard",
)


trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset = dataset["train"],
    eval_dataset = dataset["test"],
    processing_class = tokenizer,
    peft_config = LoRA_config ,
)

# LoRA params must be fp32 for the fp16 grad scaler
for p in trainer.model.parameters():
    if p.requires_grad:
        p.data = p.data.float()

In [ ]:
# Sanity check: trainable params should be float32 only
from collections import Counter

print("Model parameter dtypes:")
print(Counter(str(p.dtype) for p in trainer.model.parameters()))

print("\nTrainable parameter dtypes:")
print(Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))

In [ ]:
trainer.train()

In [ ]:
trainer.save_model("FineTuning_ExpenseClassification_5k")
tokenizer.save_pretrained("FineTuning_ExpenseClassification_5k")

In [ ]:
# Quick inference test on a held-out example
example = dataset["test"][20]
messages = example["prompt"]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True,
).to(trainer.model.device)

trainer.model.eval()
with torch.no_grad():
    out = trainer.model.generate(**inputs, max_new_tokens=128, do_sample=False)

generated = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print("Generated:", generated)
print("Expected: ", example["completion"][0]["content"])

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel

# Load Model base model merge
model = AutoModelForCausalLM.from_pretrained(model_id,dtype=torch.float16,low_cpu_mem_usage=True)


In [ ]:
merged_model = PeftModel.from_pretrained(model, args.output_dir).merge_and_unload()

In [ ]:
!df -h /content

In [ ]:
!ls -lh /content/ExpenseClassification_Finetune_merged_5k/

In [ ]:
merged_model.save_pretrained("ExpenseClassification_Finetune_merged_5k", safe_serialization=True, max_shard_size="2GB")


In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
tok.save_pretrained("ExpenseClassification_Finetune_merged_5k")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_path = "/content/ExpenseClassification_Finetune_merged_5k"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    device_map="auto",
    torch_dtype="auto"
)

print("Merged model loaded successfully!")

In [ ]:
from random import randint
from transformers import pipeline, GenerationConfig

# Generation settings. Greedy decoding (no sampling) gives the same answer every time.
# Qwen marks the end of the assistant's reply with <|im_end|>, so generation stops there.
config = GenerationConfig(
    max_new_tokens=256,
    do_sample=False,
    eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"),
    pad_token_id=tok.pad_token_id,
)

# Wrap the model and tokenizer in a text-generation pipeline
pipe = pipeline("text-generation", model=model, tokenizer=tok)

# Pick a random row from the test set (-1 because randint includes both ends)
rand_idx = randint(0, len(dataset["test"]) - 1)
test_sample = dataset["test"][rand_idx]

# Turn the system + user messages into Qwen's chat format, ending with the "assistant" cue
prompt = tok.apply_chat_template(test_sample["prompt"], tokenize=False, add_generation_prompt=True)

# return_full_text=False gives only the new answer, so no slicing of the prompt is needed
outputs = pipe(prompt, generation_config=config, return_full_text=False)

print("Expense text:\n", test_sample["prompt"][1]["content"])
print("\nOriginal answer:\n", test_sample["completion"][0]["content"])
print("\nGenerated answer:\n", outputs[0]["generated_text"].strip())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/drive/MyDrive/ExpenseProject
!cp -r /content/ExpenseClassification_Finetune_merged /content/drive/MyDrive/ExpenseProject/
!zip -qr /content/drive/MyDrive/ExpenseProject/adapter.zip ExpenseClassification_Finetune -x "*/checkpoint-*"
!ls -lh /content/drive/MyDrive/ExpenseProject

In [ ]:
save_path = "/content/drive/MyDrive/ExpenseProject/merged_model"
model.save_pretrained(save_path, safe_serialization=True)
tok.save_pretrained(save_path)
!ls -lh {save_path}

In [ ]:
from huggingface_hub import notebook_login, HfApi
notebook_login()            # a box appears: paste the token and press Login

In [ ]:
! pip install -U bitsandbytes>=0.46.1

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_id = "Prachi1802/ExpenseClassifier"      # your fine-tuned, merged model
tokenizer = AutoTokenizer.from_pretrained(model_id)
dtype = torch.float16

model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

In [ ]:
import json, re

SYSTEM_PROMPT = """You check employee expenses against company policy.
Read the expense text and reply with only a JSON object with these keys:
amount, currency, category, policy_limit_amount, reimbursement_allowed, receipt_provided, flag, reason.

Categories: Travel, Ground Transport, Lodging, Meals, Software, Office Supplies, Training, Other.
Currencies: USD, EUR, GBP, INR, CAD.

Limits (USD / EUR / GBP / INR / CAD):
- Meals, per person: 75 / 70 / 60 / 6000 / 100
- Lodging, per night: 250 / 230 / 200 / 20000 / 340
- Ground Transport, per expense: 100 / 90 / 80 / 8000 / 135
- Travel, per expense: 800 / 750 / 650 / 64000 / 1100
- Software, per expense: 200 / 180 / 160 / 16000 / 270
- Office Supplies, per expense: 150 / 140 / 120 / 12000 / 200
- Training, per expense: 1000 / 900 / 800 / 80000 / 1350
- Other, per expense: 100 / 90 / 80 / 8000 / 135
A receipt is required above 25 USD, 25 EUR, 20 GBP, 2000 INR or 35 CAD.

Rules:
- policy_limit_amount = per-person limit x attendees for Meals, per-night limit x nights for Lodging, otherwise the limit. Use null if it cannot be worked out, and 0 if the expense is not reimbursable.
- receipt_provided is "No" only if the text says the receipt is missing.
- Not reimbursable: alcohol-only purchases, fines, upgrades, travel or rooms for family or friends, personal trips, hotel extras, personal subscriptions, gift cards, personal or home items, hobby classes, personal services or purchases.
- flag, first match wins: not_reimbursable; missing_info (amount, attendees, nights or a required receipt missing); over_limit (amount above policy_limit_amount); none.
- reimbursement_allowed is "No" only when the flag is not_reimbursable.
- Use null for anything the text does not state. reason briefly explains the flag."""
USER_TEMPLATE = "Expense: {text}"

def check(text):
    chat = [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_TEMPLATE.format(text=text)}]
    inputs = tokenizer.apply_chat_template(chat, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=256, do_sample=False,
                             eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
                             pad_token_id=tokenizer.pad_token_id)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", answer, re.S)
    try:
        return json.loads(m.group(0))
    except Exception:
        return answer

print(json.dumps(check("Dinner at Nobu NYC 2 ppl 198.50 usd"), indent=2))